# MLlib RDD-Based Implementation

In [1]:
from pyspark.sql import SparkSession

from pyspark.mllib.regression import LabeledPoint
from pyspark.mllib.tree import DecisionTree
from pyspark.mllib.regression import LabeledPoint
from pyspark.mllib.linalg import Vectors

In [2]:
spark = SparkSession.builder.appName("MLlib").getOrCreate()
sc = spark.sparkContext

25/05/10 11:37:38 WARN Utils: Your hostname, dyln resolves to a loopback address: 127.0.1.1; using 192.168.1.37 instead (on interface wlp0s20f3)
25/05/10 11:37:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/10 11:37:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
lines = sc.textFile("hdfs://172.17.0.2:9000/user/khtn_22120182/train.csv")
header = lines.first()

# Get indices of numeric columns (excluding id and categorical columns)
header_cols = header.split(",")
numeric_col_indices = [1, 4, 5, 6, 7, 8, 10]  # vendor_id, passenger_count, longitudes, latitudes, trip_duration

data = lines.filter(lambda line: line != header)
parsed = data.map(lambda line: [float(line.split(",")[i]) for i in numeric_col_indices])
rdd = parsed.map(lambda cols: LabeledPoint(cols[-1], Vectors.dense(cols[:-1])))

train_data, test_data = rdd.randomSplit([0.8, 0.2], seed=0) 

## max_depth=3

In [4]:
# Train
model_rdd = DecisionTree.trainRegressor(train_data,
                                        categoricalFeaturesInfo={},
                                        impurity="variance",
                                        maxDepth=3)

25/05/10 11:37:55 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [5]:
# Evaluate
test_data_collected = test_data.collect()  # Collect test data to the driver
predictions = [(p.label, model_rdd.predict(p.features)) for p in test_data_collected]  # Perform predictions on the driver
predictions_rdd = sc.parallelize(predictions)  # Parallelize predictions back into an RDD

rmse_rdd = predictions_rdd.map(lambda lp: (lp[0] - lp[1])**2).mean() ** 0.5

mean_label = predictions_rdd.map(lambda lp: lp[0]).mean()
ss_total = predictions_rdd.map(lambda lp: (lp[0] - mean_label) ** 2).sum()
ss_residual = predictions_rdd.map(lambda lp: (lp[0] - lp[1]) ** 2).sum()
r2_rdd = 1 - (ss_residual / ss_total)

print("R2 (RDD):", r2_rdd)
print("RMSE (RDD):", rmse_rdd)
print("Tree Structure:\n", model_rdd.toDebugString())

R2 (RDD): 0.007390113874465087
RMSE (RDD): 4924.975609542068
Tree Structure:
 DecisionTreeModel regressor of depth 3 with 15 nodes
  If (feature 2 <= -73.92247772216797)
   If (feature 4 <= -73.92765045166016)
    If (feature 5 <= 40.70283126831055)
     Predict: 1436.9155253423974
    Else (feature 5 > 40.70283126831055)
     Predict: 816.4229732850874
   Else (feature 4 > -73.92765045166016)
    If (feature 5 <= 40.67620658874512)
     Predict: 2982.32323950818
    Else (feature 5 > 40.67620658874512)
     Predict: 1706.2818683904202
  Else (feature 2 > -73.92247772216797)
   If (feature 4 <= -73.95597457885742)
    If (feature 2 <= -73.837646484375)
     Predict: 2138.7198721647655
    Else (feature 2 > -73.837646484375)
     Predict: 3417.908314773571
   Else (feature 4 > -73.95597457885742)
    If (feature 3 <= 40.66048622131348)
     Predict: 1786.178858480008
    Else (feature 3 > 40.66048622131348)
     Predict: 1087.9325157485591



## max_depth=5

In [6]:
# Train
model_rdd = DecisionTree.trainRegressor(train_data,
                                        categoricalFeaturesInfo={},
                                        impurity="variance",
                                        maxDepth=5)

In [7]:
# Evaluate
test_data_collected = test_data.collect()  # Collect test data to the driver
predictions = [(p.label, model_rdd.predict(p.features)) for p in test_data_collected]  # Perform predictions on the driver
predictions_rdd = sc.parallelize(predictions)  # Parallelize predictions back into an RDD

rmse_rdd = predictions_rdd.map(lambda lp: (lp[0] - lp[1])**2).mean() ** 0.5

mean_label = predictions_rdd.map(lambda lp: lp[0]).mean()
ss_total = predictions_rdd.map(lambda lp: (lp[0] - mean_label) ** 2).sum()
ss_residual = predictions_rdd.map(lambda lp: (lp[0] - lp[1]) ** 2).sum()
r2_rdd = 1 - (ss_residual / ss_total)

print("R2 (RDD):", r2_rdd)
print("RMSE (RDD):", rmse_rdd)
print("Tree Structure:\n", model_rdd.toDebugString())

R2 (RDD): 0.007852671402200806
RMSE (RDD): 4923.82795323045
Tree Structure:
 DecisionTreeModel regressor of depth 5 with 63 nodes
  If (feature 2 <= -73.92247772216797)
   If (feature 4 <= -73.92765045166016)
    If (feature 5 <= 40.70283126831055)
     If (feature 3 <= 40.730852127075195)
      If (feature 3 <= 40.710371017456055)
       Predict: 783.7204732107939
      Else (feature 3 > 40.710371017456055)
       Predict: 1286.4743048222456
     Else (feature 3 > 40.730852127075195)
      If (feature 5 <= 40.67620658874512)
       Predict: 2190.1706995884774
      Else (feature 5 > 40.67620658874512)
       Predict: 1745.9044147428108
    Else (feature 5 > 40.70283126831055)
     If (feature 0 <= 1.5)
      If (feature 5 <= 40.72855758666992)
       Predict: 856.3953096865805
      Else (feature 5 > 40.72855758666992)
       Predict: 681.1093347596973
     Else (feature 0 > 1.5)
      If (feature 5 <= 40.72391891479492)
       Predict: 1114.0757716989926
      Else (feature 5 > 40.72

## max_depth=7

In [8]:
# Train
model_rdd = DecisionTree.trainRegressor(train_data,
                                        categoricalFeaturesInfo={},
                                        impurity="variance",
                                        maxDepth=7)

In [9]:
# Evaluate
test_data_collected = test_data.collect()  # Collect test data to the driver
predictions = [(p.label, model_rdd.predict(p.features)) for p in test_data_collected]  # Perform predictions on the driver
predictions_rdd = sc.parallelize(predictions)  # Parallelize predictions back into an RDD

rmse_rdd = predictions_rdd.map(lambda lp: (lp[0] - lp[1])**2).mean() ** 0.5

mean_label = predictions_rdd.map(lambda lp: lp[0]).mean()
ss_total = predictions_rdd.map(lambda lp: (lp[0] - mean_label) ** 2).sum()
ss_residual = predictions_rdd.map(lambda lp: (lp[0] - lp[1]) ** 2).sum()
r2_rdd = 1 - (ss_residual / ss_total)

print("R2 (RDD):", r2_rdd)
print("RMSE (RDD):", rmse_rdd)
print("Tree Structure:\n", model_rdd.toDebugString())

R2 (RDD): -0.004299327847247181
RMSE (RDD): 4953.890147573222
Tree Structure:
 DecisionTreeModel regressor of depth 7 with 251 nodes
  If (feature 2 <= -73.92247772216797)
   If (feature 4 <= -73.92765045166016)
    If (feature 5 <= 40.70283126831055)
     If (feature 3 <= 40.730852127075195)
      If (feature 3 <= 40.710371017456055)
       If (feature 2 <= -74.0030403137207)
        If (feature 4 <= -73.96220016479492)
         Predict: 994.7974330357143
        Else (feature 4 > -73.96220016479492)
         Predict: 1753.0058823529412
       Else (feature 2 > -74.0030403137207)
        If (feature 0 <= 1.5)
         Predict: 572.4636432733722
        Else (feature 0 > 1.5)
         Predict: 814.0949626416004
      Else (feature 3 > 40.710371017456055)
       If (feature 5 <= 40.67620658874512)
        If (feature 4 <= -73.98814010620117)
         Predict: 1780.7870159453303
        Else (feature 4 > -73.98814010620117)
         Predict: 1492.8500590318772
       Else (feature 5 > 40

In [10]:
spark.stop()